In [0]:
spark.conf.set("spark.sql.shuffle.partitions",30)

In [0]:
df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load('/Volumes/external-catalog/default/test-volume'))
display(df)

In [0]:
from pyspark.sql.functions import col

high_risk_df = (
    df.filter(
        (col("Attrition") == "No") &
        (col("JobSatisfaction").cast("int") < 3)
    )
)
columns = ["EmployeeNumber","EmployeeName", "Department","JobRole","JobSatisfactions","Age","Gender","MaritialStatus","MonthlyIncome","OverTime","YearsAtCompany"]
high_risk_df = high_risk_df.select(*[c for c in columns if c in high_risk_df.columns])
high_risk_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_risk_employees")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_employees")
history_df = delta_table.history()
display(history_df.select("version"))

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_employees")
display(history_df.select("version","timestamp","operation"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_employees")
display(df_delta)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import Row

schema = StructType([
    StructField("EmployeeNumber", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("JobRole", StringType(), True),
    StructField("Age", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("MonthlyIncome", IntegerType(), True),
    StructField("OverTime", StringType(), True),
    StructField("YearsAtCompany", IntegerType(), True)
])

dummy_data = [
    (999999, "Dummy Dept", "Dummy Role", "30", "Other", 0, "No", 0)
]

dummy_df = spark.createDataFrame(dummy_data, schema=schema)
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.high_risk_employees")

In [0]:
history_df = spark.sql("DESCRIBE HISTORY `external-catalog`.default.high_risk_employees")
display(history_df.select("version","timestamp","operation"))

In [0]:
df_delta = spark.read.format("delta").table("`external-catalog`.default.high_risk_employees")
display(df_delta)

In [0]:
# Get the timestamps for the first two versions of the Delta table
history_df = (
    spark.read.format("delta")
    .table("`external-catalog`.default.high_risk_employees")
    .option("readChangeFeed", "true")  # ensure we can read the log
    .option("versionAsOf", 0)  # placeholder to access the table; will be overridden
)

# Use DeltaTable API to fetch the history (more reliable)
from delta.tables import DeltaTable

delta_tbl = DeltaTable.forName(spark, "`external-catalog`.default.high_risk_employees")
versions_ts = (
    delta_tbl.history()
    .filter("version IN (0, 1)")
    .select("version", "timestamp")
    .orderBy("version")
    .collect()
)

# Map version → timestamp
ts_by_version = {row["version"]: row["timestamp"] for row in versions_ts}

# Read the table as of each timestamp
for ver in sorted(ts_by_version):
    ts = ts_by_version[ver].strftime("%Y-%m-%dT%H:%M:%S.%f%z")
    df_at_ts = (
        spark.read.format("delta")
        .option("timestampAsOf", ts)
        .table("`external-catalog`.default.high_risk_employees")
    )
    display(df_at_ts)

In [0]:
df_version_0 = spark.read.format("delta").option("versionAsOf", 0).table("`external-catalog`.default.high_risk_employees")
df_version_1 = spark.read.format("delta").option("versionAsOf", 1).table("`external-catalog`.default.high_risk_employees")
display(df_version_0)
display(df_version_1)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data")

In [0]:
from pyspark.sql.functions import col,upper

# Example logical transformation: filter employees with MonthlyIncome > 3000 and Age > 25
transformed_df = df.filter((col("MonthlyIncome") > 3000) & (col("Age") > 25))

output_path = "/Volumes/external-catalog/default/employee_transformed_data"

(transformed_df
    .write
    .mode("overwrite")
    .partitionBy("Department")
    .format("delta")
    .save(output_path)
)